In [ ]:
import os
import gc
import ot
import pickle
import anndata
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
from scipy.stats import spearmanr, pearsonr
from scipy.spatial import distance_matrix
import matplotlib.pyplot as plt
import os.path as osp
from pathlib import Path
from scipy.sparse import csr_matrix
import time
import csv
from anndata import concat
import comflow as cf
import sys
directory_dataRead_path = '/home/xiem6/0Functions'
sys.path.append(directory_dataRead_path)
import st_loading_utils

In [ ]:
## BRCA_Human_3Ddataset_5 dataset
data_name = "BRCA_5_mergeddb"
slice_names = ["2", "3"] # used to name the output file
slice_names = ["0", "1", "2", "3"] # used to name the output file
slice_names = ["0"] # used to name the output file
if_clu = 'No' # no need to do clustering because the data comes with annotation
impute = "No" # if we use imputed data or not
celltype_key = 'Celltype'
background_key = 'summary' # scatter plot with color representing total sent or received signal.
species= "human" # mouse and human has different database & maximum steps from receptor to TF
database_name= "merged" # database_name= "cellchat"
DSdatabaseN = "pathways" # downstream database
min_cell_pct = 0.05
co_exp_ratio=0.01
dis_thr = 500
out_dir = "/maiziezhou_lab3/manfeifei/0Projects/localOT/COMMOT/LOT_CCC_0903_25/result"
ct_out = "/maiziezhou_lab3/manfeifei/0Projects/localOT/COMMOT/analysis/raw_output"
out_dir = osp.join(out_dir, data_name)
os.makedirs(out_dir, exist_ok=True)
ct_out = osp.join(ct_out, data_name)
os.makedirs(ct_out, exist_ok=True)

In [ ]:
# replace "/" to "." in CT names
def sanitize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.map(lambda x: x.replace("/", ".") if isinstance(x, str) else x)
    df.columns = df.columns.map(lambda x: x.replace("/", ".") if isinstance(x, str) else x)
    return df

In [ ]:
for slice_name in slice_names:
    start = time.time()
    if data_name == "painControl75": 
        if impute == "No":
            input_dir = "/maiziezhou_lab/Datasets/ST_datasets/SNI_merged_0917/SNI_merged_0917.h5ad"
            adata = anndata.read_h5ad(input_dir)
            adata = adata[adata.obs["Section ID"]==slice_name].copy()
        else:
            input_dir = "/maiziezhou_lab/Datasets/ST_datasets/SNI_merged_0917"
            adata = anndata.read_h5ad(osp.join(input_dir, slice_name+"_Tangram_impute.h5ad"))
        adata.obsm["spatial"] = adata.obs[["center_x", "center_y"]].to_numpy()
    elif data_name == "spatialZ_painControl":
        if "Sham" in slice_name:
            dst = "/maiziezhou_lab3/manfeifei/0Projects/localOT/COMMOT/spatialZ/Sham_interpolated_9slices_alignedXY.h5ad"
        else:
            dst = "/maiziezhou_lab3/manfeifei/0Projects/localOT/COMMOT/spatialZ/SNI_interpolated_9slices_alignedXY.h5ad"
        adata = sc.read_h5ad(dst)
        # Unique slice IDs in their order of appearance
        slice_ids = adata.obs["slice_id"].unique()

        # z-coordinate for each slice
        z_values = np.arange(0, 251, 25)

        assert len(slice_ids) == len(z_values), \
            f"Expected 11 slices, found {len(slice_ids)}"

        # Map slice_id -> z
        slice_to_z = dict(zip(slice_ids, z_values))

        # Get z value for every spot
        z = adata.obs["slice_id"].map(slice_to_z).to_numpy()

        # Assuming spatial currently contains [x, y]
        xy = adata.obsm["spatial"][:, :2]

        # Create [x, y, z]
        adata.obsm["spatial"] = np.column_stack([xy, z])
    elif data_name == "SNI_L": 
        if impute == "No":
            input_dir = "/maiziezhou_lab/Datasets/ST_datasets/SNI_merged_0917/SNI_merged_0917.h5ad"
            adata = anndata.read_h5ad(input_dir)
            adata = adata[adata.obs["Section ID"]==slice_name].copy()
        else:
            input_dir = "/maiziezhou_lab/Datasets/ST_datasets/SNI_merged_0917"
            adata = anndata.read_h5ad(osp.join(input_dir, slice_name+"_Tangram_impute.h5ad"))
        adata.obsm["spatial"] = adata.obs[["center_x", "center_y"]].to_numpy()
    elif data_name == "DLPFC":
        section_file = "/maiziezhou_lab/Datasets/ST_datasets/DLPFC12"
        adata = st_loading_utils.load_DLPFC(root_dir=section_file, section_id=slice_name)
        adata.obs['Ground Truth'] = adata.obs['original_clusters'].astype('category')
    elif data_name == "BRCA_5":
        section_file = "/maiziezhou_lab4/Datasets/ST_datasets/BRCA_Human_3Ddataset_5/BRCA_Human_3Ddataset_5.h5ad"
        adata = anndata.read_h5ad(section_file) 
        adata.obs["slice"] = adata.obs_names.str.split("-").str[-1]
        adata=adata[adata.obs["slice"] == 'slice'+slice_name]
    elif data_name == "BRCA_5_3D_So3D":
        section_file = "/maiziezhou_lab4/Datasets/ST_datasets/BRCA_Human_3Ddataset_5/BRCA_Human_3Ddataset_5.h5ad"
        adata = anndata.read_h5ad(section_file) 
        sp = adata.obsm["spatial"].copy()
        mapping = {0.0: 0.0, 19.84: 100.0, 39.68: 200.0, 59.52: 300.0}
        col3 = sp[:, 2]
        sp[:, 2] = np.vectorize(mapping.get)(col3)
        adata.obsm["spatial"] = sp
    elif data_name in ("BRCA_5_3D", "BRCA_5_3D_GT"):
        section_ids = ["0", "1", "2", "3"]
        if data_name == "BRCA_5_3D":
            coord_path = "/maiziezhou_lab/manfeifei/0Projects/localOT/BRCA_5_CT/1/threeD/aligned_spatial.csv"
        else:
            coord_path = "/maiziezhou_lab/manfeifei/0Projects/localOT/BRCA_5_GT/1/threeD/aligned_spatial.csv"

        z_map = {
            "0": 0,
            "1": 100,
            "2": 200,
            "3": 300,
        }
        # -------------------------
        # Load aligned coordinate file
        # -------------------------
        coord_df = pd.read_csv(coord_path)

        # cell_name example: AAACAAGTATCTCCCA-1_151673
        coord_df["slice"] = coord_df["cell_name"].str.rsplit("_", n=1).str[1]
        coord_df["spot_name"] = coord_df["cell_name"].str.rsplit("_", n=1).str[0]

        adata_list = []

        # -------------------------
        # Load each slice and match spots slice-by-slice
        # -------------------------
        section_file = "/maiziezhou_lab4/Datasets/ST_datasets/BRCA_Human_3Ddataset_5/BRCA_Human_3Ddataset_5.h5ad"
        adata4 = anndata.read_h5ad(section_file) 
        adata4.obs["slice"] = adata4.obs_names.str.split("-").str[-1]
        
        for section_id in section_ids:
            print(f"\nProcessing slice {section_id}")
            adata=adata4[adata4.obs["slice"] == 'slice'+section_id]
            adata.obs["slice"] = section_id

            # Coordinates for this slice only
            coord_sub = coord_df[coord_df["slice"] == section_id].copy()
            coord_sub = coord_sub.set_index("spot_name")

            # Match spots within this slice
            common_spots = adata.obs_names.intersection(coord_sub.index)
            print(f"Matched spots: {len(common_spots)} / {adata.n_obs}")

            # Keep only matched spots
            adata = adata[common_spots].copy()
            # Reorder coord_sub to exactly match adata.obs_names
            coord_sub = coord_sub.loc[adata.obs_names].copy()
            # Replace Z with fixed slice-specific value
            coord_sub["Z"] = z_map[section_id]
            # Add aligned 3D coordinates to obsm
            adata.obsm["spatial"] = coord_sub[["X", "Y", "Z"]].to_numpy()
            adata.obsm["spatial_3D_aligned"] = coord_sub[["X", "Y", "Z"]].to_numpy()
            # Optional: save coordinates and labels in obs
            adata.obs["aligned_X"] = coord_sub["X"].values
            adata.obs["aligned_Y"] = coord_sub["Y"].values
            adata.obs["aligned_Z"] = coord_sub["Z"].values
            adata.obs["gt_labels_from_alignment"] = coord_sub["gt_labels"].astype(str).values
            # adata.obs["celltype_colors"] = coord_sub["celltype_colors"].values
            adata_list.append(adata)
        # -------------------------
        # Concatenate all slices
        # -------------------------
        adata_all = concat(
            adata_list,
            axis=0,
            join="outer",
            label="batch",
            keys=section_ids,
            index_unique="_"
        )

        # -------------------------
        # Sanity checks
        # -------------------------
        print("\nFinal concatenated AnnData:")
        print(adata_all)

        print("\nSpatial shape:")
        print(adata_all.obsm["spatial"].shape)

        print("\nZ values by slice:")
        for section_id in section_ids:
            z_vals = np.unique(
                adata_all.obs.loc[adata_all.obs["slice"] == section_id, "aligned_Z"]
            )
            print(section_id, z_vals)

        print("\nFirst 5 spatial coordinates:")
        print(adata_all.obsm["spatial"][:5])
        adata = adata_all.copy()
    elif data_name == "MERFISH_mouse_spinalcord":
        input_dir = "/maiziezhou_lab/Datasets/ST_datasets/MERFISH_mouse_spinalcord/MERFISH_spinal_cord_resolved_0718.h5ad"
        adata = anndata.read_h5ad(input_dir)
        ID_key = "Section ID"
        adata = adata[adata.obs[ID_key] == slice_name]
    elif data_name == "lymph":
        input_dir = "/maiziezhou_lab/Datasets/ST_datasets/humanMetastaticLymphNode/GSE251926_metastatic_lymph_node_3d.h5ad"
        adata = sc.read_h5ad(input_dir)
        adata = adata[adata.obs['n_section'] == slice_name].copy()
        slice_name = str(slice_name)
    elif data_name == "embryo":
        input_dir = "/maiziezhou_lab/Datasets/ST_datasets/MOSTA_embryo"
        adata = st_loading_utils.load_Mostaembryo(root_dir=input_dir, section_id=slice_name)
    elif data_name == "GC":
        datapath = "/maiziezhou_lab4/Datasets/HwangLab_Data/JapanNCCE_GC_Pre-Post_ICI_06-2023/spaceranger_outs/NCCE_"
        ct_path = "/maiziezhou_lab3/manfeifei/0Projects/ST_CNV/CT_annotation/JapanNCCE/JapanNCCE_finalCT"
        dir_input = os.path.join(datapath + slice_name, "outs")
        adata = sc.read_visium(path=dir_input)

        dup = adata.var_names[adata.var_names.duplicated()]
        print("duplicated genes:", len(dup))
        adata.var_names_make_unique()

        ct_file = os.path.join(ct_path, f"{slice_name}.csv")
        ct_df = pd.read_csv(ct_file)

        ct_df["pixel"] = ct_df["pixel"].astype(str)

        ct_map = dict(zip(ct_df["pixel"], ct_df["cell_type"]))
        adata.obs["cell_type"] = adata.obs_names.map(ct_map)

        print(f"Matched spots   : {adata.obs['cell_type'].notna().sum()} / {adata.n_obs}")
        print(f"Unknown spots   : {adata.obs['cell_type'].isna().sum()}")

        adata.obs["cell_type"] = adata.obs["cell_type"].fillna("Unknown")
        adata.obs["cell_type"] = adata.obs["cell_type"].replace(
            ["nan", "NaN", "NA", "None", ""],
            "Unknown"
        )
        adata.obs["cell_type"] = adata.obs["cell_type"].astype(str)
    elif data_name == "GC_CNV":
        datapath = "/maiziezhou_lab4/Datasets/HwangLab_Data/JapanNCCE_GC_Pre-Post_ICI_06-2023/spaceranger_outs/NCCE_"
        ct_path = "/maiziezhou_lab3/manfeifei/0Projects/ST_CNV/CT_annotation/JapanNCCE/JapanNCCE_finalCT"
        dir_input = os.path.join(datapath + slice_name, "outs")
        adata = sc.read_visium(path=dir_input)

        dup = adata.var_names[adata.var_names.duplicated()]
        print("duplicated genes:", len(dup))
        adata.var_names_make_unique()

        ct_file = os.path.join(ct_path, f"{slice_name}.csv")
        ct_df = pd.read_csv(ct_file)

        ct_df["pixel"] = ct_df["pixel"].astype(str)

        ct_map = dict(zip(ct_df["pixel"], ct_df["cell_type"]))
        adata.obs["cell_type"] = adata.obs_names.map(ct_map)

        print(f"Matched spots   : {adata.obs['cell_type'].notna().sum()} / {adata.n_obs}")
        print(f"Unknown spots   : {adata.obs['cell_type'].isna().sum()}")

        adata.obs["cell_type"] = adata.obs["cell_type"].fillna("Unknown")
        adata.obs["cell_type"] = adata.obs["cell_type"].replace(
            ["nan", "NaN", "NA", "None", ""],
            "Unknown"
        )
        # adata.obs["cell_type"] = adata.obs["cell_type"].astype(str)
        adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
        CNV_dir = "/maiziezhou_lab/manfeifei/0Projects/localOT/COMMOT/copyKAT"
        if "tumor" not in adata.obs["cell_type"].cat.categories:
            adata.obs["cell_type"] = adata.obs["cell_type"].cat.add_categories(["tumor"])
        cnv_path = os.path.join(CNV_dir, "NCCE_"+slice_name, "prediction_aligned.txt")
        cnv = pd.read_csv(cnv_path, sep="\t", comment="#")

        # build mapping: cell.names -> copykat.pred
        cnv["cell.names"] = cnv["cell.names"].astype(str)
        cnv_map = cnv.set_index("cell.names")["copykat.pred"].to_dict()

        # attach CopyKAT prediction to adata
        adata.obs["copykat_pred"] = adata.obs_names.astype(str).map(cnv_map)

        print("CopyKAT matched:", adata.obs["copykat_pred"].notna().sum(), "/", adata.n_obs)

        # change epithelial -> tumor if copykat_pred == aneuploid
        mask = (adata.obs["cell_type"] == "epithelial") & (adata.obs["copykat_pred"] == "aneuploid")
        print("Epithelial->tumor:", mask.sum())

        adata.obs.loc[mask, "cell_type"] = "tumor"
        adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")

    elif data_name == "DLPFC_3D":
        section_file = "/maiziezhou_lab/Datasets/ST_datasets/DLPFC12"
        section_ids = ["151673", "151674", "151675", "151676"]
        coord_path = "/maiziezhou_lab/manfeifei/0Projects/localOT/for_DLPFC_realData/3D/spacel_Clu_Vclu_dis_staAll_0105_26/1/151673_151676/localOT_DLPFC_151673_151676_aligned_coordinates_colors.csv"

        z_map = {
            "151673": 0,
            "151674": 10,
            "151675": 310,
            "151676": 320,
        }
        # -------------------------
        # Load aligned coordinate file
        # -------------------------
        coord_df = pd.read_csv(coord_path)

        # cell_name example: AAACAAGTATCTCCCA-1_151673
        coord_df["slice"] = coord_df["cell_name"].str.rsplit("_", n=1).str[1]
        coord_df["spot_name"] = coord_df["cell_name"].str.rsplit("_", n=1).str[0]

        adata_list = []

        # -------------------------
        # Load each slice and match spots slice-by-slice
        # -------------------------
        for section_id in section_ids:
            print(f"\nProcessing slice {section_id}")

            adata = st_loading_utils.load_DLPFC(
                root_dir=section_file,
                section_id=section_id
            )

            adata.obs["Ground Truth"] = adata.obs["original_clusters"].astype("category")
            adata.obs["slice"] = section_id

            # Coordinates for this slice only
            coord_sub = coord_df[coord_df["slice"] == section_id].copy()
            coord_sub = coord_sub.set_index("spot_name")

            # Match spots within this slice
            common_spots = adata.obs_names.intersection(coord_sub.index)
            print(f"Matched spots: {len(common_spots)} / {adata.n_obs}")

            # Keep only matched spots
            adata = adata[common_spots].copy()
            # Reorder coord_sub to exactly match adata.obs_names
            coord_sub = coord_sub.loc[adata.obs_names].copy()
            # Replace Z with fixed slice-specific value
            coord_sub["Z"] = z_map[section_id]
            # Add aligned 3D coordinates to obsm
            adata.obsm["spatial"] = coord_sub[["X", "Y", "Z"]].to_numpy()
            adata.obsm["spatial_3D_aligned"] = coord_sub[["X", "Y", "Z"]].to_numpy()
            # Optional: save coordinates and labels in obs
            adata.obs["aligned_X"] = coord_sub["X"].values
            adata.obs["aligned_Y"] = coord_sub["Y"].values
            adata.obs["aligned_Z"] = coord_sub["Z"].values
            adata.obs["gt_labels_from_alignment"] = coord_sub["gt_labels"].astype(str).values
            adata.obs["celltype_colors"] = coord_sub["celltype_colors"].values
            adata_list.append(adata)
        # -------------------------
        # Concatenate all slices
        # -------------------------
        adata_all = concat(
            adata_list,
            axis=0,
            join="outer",
            label="batch",
            keys=section_ids,
            index_unique="_"
        )

        # -------------------------
        # Sanity checks
        # -------------------------
        print("\nFinal concatenated AnnData:")
        print(adata_all)

        print("\nSpatial shape:")
        print(adata_all.obsm["spatial"].shape)

        print("\nZ values by slice:")
        for section_id in section_ids:
            z_vals = np.unique(
                adata_all.obs.loc[adata_all.obs["slice"] == section_id, "aligned_Z"]
            )
            print(section_id, z_vals)

        print("\nFirst 5 spatial coordinates:")
        print(adata_all.obsm["spatial"][:5])
        adata = adata_all.copy()
    elif data_name == "Kidney":
        adata= adata_[adata_.obs["Case"] == slice_name].copy()
    elif data_name == "E155_FEAST_3D":
        dst = "/maiziezhou_lab3/manfeifei/0Projects/FEAST/data/E15.5_FEAST_3D.h5ad"
        adata_= sc.read_h5ad(dst)
        adata = adata_.copy()
        adata.obsm['spatial'] = adata_.obsm['spatial_3d']
    else: #data_name = "V1_Mouse_Brain_Sagittal_Posterior"
        adata = sc.datasets.visium_sge(sample_id='V1_Mouse_Brain_Sagittal_Posterior')

    print("slice:, ", adata)
    print("slice size:", adata.shape)

    if if_clu == "Yes":
        Clu_key = 'leiden'
    else:
        Clu_key = celltype_key
    """# **Visium data example (mouse brain)**

    ## **Preprocessing the data**
    Here, **non-negative** values that reasonably reflect the abundancy of molecules are needed. \
    We will perform the most basic data preprocessing here with only normalizing total count and log1p transform. \
    A more detailed preprocessing could be performed such as regressing out cell cycle genes.
    """

    adata.var_names_make_unique()
    adata.raw = adata
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)

    adata_dis500 = adata.copy()

    """Perform a basic clustering of the data using common methods for scRNA-seq data.
    Alternatively, tools dedicated for spatial transcriptomics data could be used, for example, SpaceFlow (Ren, Honglei, et al. "Identifying multicellular spatiotemporal organization of cells with SpaceFlow." Nature communications 13.1 (2022): 1-14.).
    """

    if if_clu == "Yes":
        sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        adata = adata[:, adata.var.highly_variable]
        sc.tl.pca(adata, svd_solver='arpack')
        sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
        sc.tl.umap(adata)
        sc.tl.leiden(adata, resolution=0.4)
        sc.pl.spatial(adata, color='leiden')

    """## **Spatial communication inference**

    We will use the CellChatDB ligand-receptor database here. Only the secreted signaling LR pairs will be used.

    Jin, Suoqin, et al. "Inference and analysis of cell-cell communication using CellChat." Nature communications 12.1 (2021): 1-20.
    """
    if database_name in ['cellchat', "cellphoneDB", "merged"]:
        if database_name == "cellchat":
            databaseN = "CellChat"
        elif database_name == "cellphoneDB":
            databaseN = "CellPhoneDB_v4.0"
        elif database_name == "merged":
            databaseN = "merged"
        df_cellchat = cf.pp.ligand_receptor_database(species=species, signaling_type='Secreted Signaling', database=databaseN)
        print(df_cellchat.shape)
        """We then filter the LR pairs to keep only the pairs with both ligand and receptor expressed in at least 5% of the spots."""
        df_cellchat_filtered = cf.pp.filter_lr_database(df_cellchat, adata_dis500, min_cell_pct=min_cell_pct)
        df_filtered = df_cellchat_filtered
    else:
        file_path = "/maiziezhou_lab/manfeifei/0Softwares/COMMOT_LOT/commot_LOT"
        if species == "human":
            file_path = osp.join(file_path, "merged_lrpairs_human.csv")
        else:
            #file_path = osp.join(file_path, "merged_lrpairs_mouse.csv") 
            file_path = osp.join(file_path, "CellChat_DeepTalk_SpaTalk_lr_mouse.csv") 
        df_User = pd.read_csv(file_path)
        # Drop the first column
        df_User = df_User.drop(columns=["Unnamed: 0"])

        print(df_User.shape)
        #df_User_filtered = cf.pp.filter_lr_database(df_User, adata_dis500, min_cell_pct=0.0001)
        df_User_filtered = cf.pp.filter_lr_database(df_User, adata_dis500, min_cell_pct=min_cell_pct)
        df_filtered = df_User_filtered

    print("filtered db, size:", df_filtered.shape)
    print("first few rows from filtered db:", df_filtered.head())
    if df_filtered.empty:
        break

    if DSdatabaseN == "pathways":
        pathways = cf.pp.downstream_database(DSdatabase=DSdatabaseN)
        print("pathway, first few rows from downstream pathways:", pathways.head())
        print("first few rows from downstream pathways:", pathways.head())
    """Now perform spatial communication inference for these 250 ligand-receptor pairs with a spatial distance limit of 500. CellChat database considers heteromeric units. The signaling results are stored as spot-by-spot matrices in the obsp slots. For example, the score for spot i signaling to spot j through the LR pair can be retrieved from `adata_dis500.obsp['comflow-cellchat-Wnt4-Fzd4_Lrp6'][i,j]`."""


    # Construct CCC networks
    # Use collective optimal transport to construct CCC networks for the ligand-receptor pairs with a spatial distance constraint of 200 (coupling between cells with distance greater than 200 is prohibited). For example, the spot-by-spot matrix for the pair Tgfb1 (ligand) and Tgfbr1_Tgfbr2 (receptor)is stored in adata.obsp['comflow-user_database-Tgfb1-Tgfbr1_Tgfbr2']. The total sent or received signal for each pair is stored in adata.obsm['comflow-user_database-sum-sender'] and adata.obsm['comflow-user_database-sum-receiver'].

    # adata_dis500,model = _spatial_communication.spatial_communication(adata_dis500,species,
    #     database_name=database_name, df_ligrec=df_filtered, pathways=pathways, co_exp_ratio=0.001, dis_thr=500, heteromeric=True, pathway_sum=True, copy = True)
    if species == "human":
        species = "Human"
    else:
        species = "Mouse" # this is because the species name in the downstream database is capitalized
    adata_dis500,model = cf.tl.spatial_communication(adata_dis500,species,
        database_name=database_name, df_ligrec=df_filtered, pathways=pathways, co_exp_ratio=co_exp_ratio, dis_thr=dis_thr, heteromeric=True, pathway_sum=True, copy = True)
    
    if if_clu == "Yes":
        adata_dis500.obs['leiden'] = adata.obs['leiden']
        
    end = time.time()
    runtime = end - start

    if data_name in ["SNI_L", "painControl75", "spatialZ_painControl"]  and impute=="Yes":
        slice_name = slice_name+"_Tangram_impute"

    if data_name == "Kidney":
        slice_namefile = sampleID + "_"+slice_name
    else:
        slice_namefile = slice_name
            # Write to CSV
    timepath = osp.join(out_dir, slice_namefile+"_runtime.csv")
    with open(timepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['slice_name', 'runtime_seconds'])
        writer.writerow([slice_namefile, runtime])

    print("\n=== obsm ===")
    for k, v in adata_dis500.obsm.items():
        print(k)
        print("  type:", type(v))
        print("  shape:", getattr(v, "shape", None))

        if isinstance(v, pd.DataFrame):
            print("  n_columns:", len(v.columns))
            print("  column chars:",
                sum(len(str(c)) for c in v.columns))
            
    key = "comflow-" + database_name + "-sum-sender"
    x = adata_dis500.obsm[key]

    print("Type:", type(x))
    print("Shape:", x.shape)

    if isinstance(x, pd.DataFrame):
        print("\n=== First 20 column names ===")
        print(x.columns[:20].tolist())

        print("\n=== Last 20 column names ===")
        print(x.columns[-20:].tolist())

        print("\n=== First 5 rows x first 10 columns ===")
        print(x.iloc[:5, :10])

        print("\n=== Column name lengths ===")
        for col in x.columns[:20]:
            print(f"{len(str(col)):4d}  {col}")
            
        # Fix large DataFrame column metadata in obsm before writing
    for key in list(adata_dis500.obsm.keys()):
        if key in ["comflow-"+database_name+"-sum-sender", "comflow-"+database_name+"-sum-receiver"]:
            print("Fixing:", key)

            obj = adata_dis500.obsm[key]

            # Convert DataFrame to numpy array to avoid huge HDF5 column-order attribute
            if isinstance(obj, pd.DataFrame):
                adata_dis500.uns[key + "_columns"] = obj.columns.astype(str).tolist()
                adata_dis500.uns[key + "_index"] = obj.index.astype(str).tolist()
                adata_dis500.obsm[key] = obj.to_numpy()    
    adata_dis500.write(osp.join(out_dir, slice_namefile+"_adata.h5ad"))

    key_df = 'comflow-'+database_name+"-info"
    lr_df = adata_dis500.uns[key_df]['df_ligrec']

    nonzero_results = []

    for idx, row in lr_df.iterrows():
        lig, rec = row['ligand'], row['receptor']
        key = f'comflow-{database_name}-{lig}-{rec}'
        
        if key in adata_dis500.obsp:
            # mat = adata_dis500.obsp[key].A if hasattr(adata_dis500.obsp[key], "A") else adata_dis500.obsp[key]
            
            # # if np.any(mat != 0):
            # #     result = 1 # has nonzeros
            # # else:
            # #     result = 0 # all zeros
            # if np.isnan(mat).any():
            #     result = "NaN"
            # elif np.any(mat != 0):
            #     result = 1   # has nonzeros
            # else:
            #     result = 0   # all zeros
            mat = adata_dis500.obsp[key]
            values = mat.data if sparse.issparse(mat) else mat
            
            # if np.any(mat != 0):
            #     result = 1 # has nonzeros
            # else:
            #     result = 0 # all zeros
            if np.isnan(values).any():
                result = "NaN"
            elif np.any(values != 0):
                result = 1   # has nonzeros
            else:
                result = 0   # all zeros
        else:
            result = "Not found"
        
        # name = key.replace("comflow-User_db-", "", 1)  # remove prefix only once
        name = lig+"--"+rec
        nonzero_results.append((name, result))

    # Convert to DataFrame
    results_df = pd.DataFrame(nonzero_results, columns=["Key", "Status"])

    # Count summary
    summary = results_df["Status"].value_counts().rename_axis("Status").reset_index(name="Count")

    # Save results
    results_df.to_csv(osp.join(out_dir, slice_namefile+"_all_LR_status_1nonZero_0allZeros.csv"), index=False)# each key status
    summary.to_csv(osp.join(out_dir, slice_namefile+"_summary_LR_counts.csv"), index=False)      # aggregated summary

    print("Detailed results saved to all_keys_status.csv")
    print("Summary saved to summary_counts.csv")
    print(summary)

    print("model.recs", model.recs)
    print(model.Q)
    df = pd.DataFrame(model.Q, columns=model.recs)
    df.to_pickle(out_dir+"/"+slice_namefile+"_model_Q.pkl")

    
    df2path = osp.join(out_dir+"/"+slice_namefile+"_rec_tf_probability.pkl")
    with open(df2path, "wb") as f:
        pickle.dump(model.receptor_tf_tables, f)

    df3path = osp.join(out_dir+"/"+slice_namefile+"_rec_tf_score.pkl")
    with open(df3path, "wb") as f:
        pickle.dump(model.rec_tf_score_tables, f)

    Cpath = out_dir+"/"+slice_namefile+"_model_C.npz"
    sparse.save_npz(Cpath, model.C)

    apath = out_dir+"/"+slice_namefile+"_model_a.npz"
    sparse.save_npz(apath, csr_matrix(model.S))
    bpath = out_dir+"/"+slice_namefile+"_model_b.npz"
    sparse.save_npz(apath, csr_matrix(model.D))

    # cal. CT level results running all LRs together
    cf.tl.cluster_communication(adata_dis500, database_name=database_name, clustering=Clu_key,
    n_permutations=100)

    # running for each LR
    key_df = 'comflow-'+database_name+"-info"
    lr_df = adata_dis500.uns[key_df]['df_ligrec']
    rows = []
    for idx, row in lr_df.iterrows():
        lig, rec = row['ligand'], row['receptor']
        key = f'comflow-{database_name}-{lig}-{rec}'
        cf.tl.cluster_communication(adata_dis500, database_name=database_name, lr_pair =[lig, rec], clustering=Clu_key,
        n_permutations=100) 
        key_name = f'comflow_cluster-{Clu_key}-{database_name}-{lig}-{rec}'
        comm_data = adata_dis500.uns[key_name]
        pval_df  = comm_data.get('communication_pvalue', None)
        score_df = comm_data.get('communication_matrix', None)  # adjust if your score key differs
        if pval_df is None or score_df is None:
            continue

        # ensure same ordering
        score_df = score_df.loc[pval_df.index, pval_df.columns]

        tmp = (
            pval_df.stack().rename("pvalue").to_frame()
            .join(score_df.stack().rename("score"))
            .reset_index()
            .rename(columns={"level_0": "sender_celltype", "level_1": "receiver_celltype"})
        )
        tmp.insert(0, "ligand", lig)
        tmp.insert(1, "receptor", rec)

        rows.append(tmp)

        ## replace "/" to "." in CT names
        if isinstance(comm_data.get("communication_matrix", None), pd.DataFrame):
            comm_data["communication_matrix"] = sanitize_df(comm_data["communication_matrix"])

        if isinstance(comm_data.get("communication_pvalue", None), pd.DataFrame):
            comm_data["communication_pvalue"] = sanitize_df(comm_data["communication_pvalue"])

        adata_dis500.uns[key_name] = comm_data  # store back (safe)


    result_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["ligand","receptor","sender_celltype","receiver_celltype","score","pvalue"]
    )

    for k, v in list(adata_dis500.uns.items()):
        if isinstance(v, dict):
            for kk in ["communication_matrix", "communication_pvalue"]:
                if isinstance(v.get(kk), pd.DataFrame):
                    v[kk] = sanitize_df(v[kk])
            adata_dis500.uns[k] = v

    adata_dis500.write(osp.join(out_dir, slice_namefile+"_adata.h5ad"))

    ct_out_slice = osp.join(ct_out, slice_namefile)
    os.makedirs(ct_out_slice, exist_ok=True)
    
    if "3D" in data_name:
        ct_out_slice = osp.join(ct_out_slice, slice_namefile+"__3DComFlow.csv")
    else:
        ct_out_slice = osp.join(ct_out_slice, slice_namefile+"__2DComFlow.csv")
    result_df.to_csv(ct_out_slice, index=False)